# 00 接続準備と保存
目的：設定と保存を確認し、許可後に短い問い合わせを1回行う。場面：導入・接続確認、目安5分（初回依存取得時間を除く）。GPU不要。

**配布前草稿。P01は採用済み。範囲・権利・経路は確認待ち。既定では外部APIを呼びません。** 有料になる操作は後半の「送信」だけ。質問は空欄、隠れた指示はありません。

公開前はローカルで`uv run python scripts/build_colab_bundle.py`を実行し、作ったZIPとこのnotebookをColabへアップロード。ZIPは`/content`へ置き、次のセルの依存取得を必要なときだけ有効にします。00～02は同じZIPで使えます。新規Colabの実機動作は未確認です。

講義用APIの利用期限は2026年11月14日23:59（日本時間）。講義の基本的な使い方を想定し、期間全体で1人10 USD分の予算を確保しています。学生全体の合計使用量には上限があります。APIキー・接続情報は専用の入力欄で扱い、Webサイト・Slack・GitHub・他のAIサービスへアップロードせず、notebook本文・出力・提出PDFにも含めないでください。詳細は同梱教材のdocs/api_usage.md。

In [ ]:
from pathlib import Path
import sys
import zipfile
import subprocess
# Colabでは左のファイル欄からseminar-colab.zipを/contentへアップロード。
# 自分が受け取った教員配布ZIPだけを展開する。パスの逸脱は拒否する。
if Path('/content').exists():
    archive = Path('/content/seminar-colab.zip')
    if not Path('/content/seminar/src').exists():
        with zipfile.ZipFile(archive) as z:
            for member in z.namelist():
                dest = (Path('/content') / member).resolve()
                if not dest.is_relative_to(Path('/content/seminar')):
                    raise ValueError('ZIP内のパスを確認してください。')
            z.extractall('/content')
    ROOT = Path('/content/seminar')
else:
    ROOT = Path.cwd()
    if not (ROOT / 'src/seminar_lab').exists():
        ROOT = ROOT.parent
INSTALL_DEPENDENCIES = False  # 新規ColabでのみTrue。固定依存の取得でありAPI送信ではない。
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--require-hashes', '-r', str(ROOT / 'requirements-colab.txt')], check=True)
sys.path.insert(0, str(ROOT / 'src'))
from seminar_lab.config import load_yaml, material_context
from seminar_lab.client import Session, LivePermission, OpenAITransport, output_text
from seminar_lab.ui import activity_view, preview_payload, save_pair
from seminar_lab.records import new_record, load_records
from seminar_lab.inputs import text_input, image_input, pdf_input
course = load_yaml(ROOT / 'config/course.yaml')
manifest = load_yaml(ROOT / 'materials/manifest.yaml')
print('ローカル設定読込み完了。まだAPIへ送信していません。')

## APIなし：個人記録を準備
共通課題の題材選択はありません。P01はPaper2AgentのNature掲載版。共通範囲はAbstract p1とFig.1a/b・caption p3に確定しています（Q1 YES）。次の空記録は送信結果ではありません。

In [ ]:
view = activity_view(course, manifest, 'P0')
print(view)
if 'offline_record' not in globals():
    offline_record = new_record(material_context(course, manifest, 'P0', for_input=False))
my_explanation = None  # 自分の説明を文字列で記入。Noneは前の記入を保持。
my_uncertainty = None
SAVE_OFFLINE_RECORD = False
if SAVE_OFFLINE_RECORD:
    if my_explanation is not None:
        offline_record['student_explanation'] = my_explanation
    if my_uncertainty is not None:
        offline_record['unresolved_point'] = my_uncertainty
    paths = save_pair([offline_record], ROOT / 'outputs')
    print([p.name for p in paths])
    assert load_records(paths[0]) == [offline_record]

## 対話の準備（送信なし）
実行経路の設定は教員から受け取る非公開ファイルを指定します。公開の空設定では送れません。
講義用はcourse_proxy、外部自習はdirect_openaiを明示的に選びます。モデル、対応設定、保持方針、講義の予算停止はM2で確認後に固定します。

初回質問欄は空欄。共通system/developer指示は追加していません。新規会話が原則のP2aではmodeをnewにします。

In [ ]:
from uuid import uuid4
runtime_path = ROOT / 'config/runtime.example.yaml'  # 許可された非公開設定へ変更
runtime = load_yaml(runtime_path)
activity_id = 'P1'  # P2a等へ切替可。共通課題の素材は自動的にP01
print(activity_view(course, manifest, activity_id))
if 'session' not in globals():
    session = Session()
if 'run_id' not in globals():
    run_id = str(uuid4())  # このセルを再実行してもIDは維持する
question = ''
conversation_mode = 'new'  # 'continue'は成功済み会話の継続
parameters = {'max_output_tokens': 400}  # 提案値。経路の確認済み範囲以外は拒否
inputs = []

In [ ]:
# P01の許可済み範囲を自分で抜粋する。質問の代わりとなる完成指示は追加しない。
source_excerpt = ''
source_location = ''
inputs = [text_input(source_excerpt, source_location)] if source_excerpt.strip() else []

## 送信前の確認
実際の質問・全履歴を確認します。画像/PDFのbase64だけ表示上省略されるので、元ファイルの見た目も開いて確認してください。入力方式の変更や履歴切捨ては自動で行いません。

In [ ]:
if question.strip():
    context = (homework_context if activity_id == 'HW1' else material_context(course, manifest, activity_id))
    print('資料と版・指定範囲:', context['material_id'], context.get('source_version'), context.get('assigned_scope'))
    print('実入力の由来:', [x.provenance for x in inputs])
    print(preview_payload(session.preview(runtime, question, inputs, conversation_mode, parameters)))
else:
    print('質問欄は空です。自分の問いを入力してください。送信していません。')

## 送信（有料・許可後だけ）
教員等の明示許可を受け、対象・USD上限・認証経路・回数を記録したLivePermissionを別途設定します。キーがあるだけでは許可になりません。講義用本キーは使いません。`SEND=True`だけではpermission未設定のため送れません。

タイムアウト時の課金は不明。失敗時に別モデルや別入力へ切り替えません。再試行は自動0回。回数枠は費用上限の保証ではありません。

In [ ]:
SEND = False
if 'permission' not in globals():
    permission = None  # 教員が案内した対象・上限・認証経路の許可を別セルで設定
if SEND:
    if not isinstance(permission, LivePermission):
        raise ValueError('対象・上限・認証経路を含む許可が必要です。')
    context = (homework_context if activity_id == 'HW1' else material_context(course, manifest, activity_id))
    # キー入力より先に、素材・設定・送信内容を検証
    session.preview(runtime, question, inputs, conversation_mode, parameters)
    from getpass import getpass
    transport = OpenAITransport(runtime, getpass('許可された個別APIキー（表示・保存しません）: '), permission)
    try:
        result = session.run(runtime=runtime, context=context, prompt=question, inputs=inputs,
                             mode=conversation_mode, parameters=parameters, run_id=run_id, transport=transport)
    finally:
        transport.close()
    print('状態:', result['status'])
    if result.get('error_message'):
        print(result['error_message'])
    print(output_text(result))
    print('費用:', result['cost_value'] if result['cost_value'] is not None else '不明')
else:
    print('送信OFF。APIは呼び出していません。')

## 本人の説明と保存
生回答はresponse_rawに残し、本人の説明・修正文は別に書きます。保存は明示操作で、上書きしない名前を作ります。Colabを閉じる・再起動する前に左右両ファイルをダウンロードしてください。出力には対話があるので公開しません。

In [ ]:
# Noneは前の記入を保持。文字列を入れた欄だけ更新する。
annotations = {
    'student_explanation': None,
    'evidence_location': None,
    'student_revision': None,
    'unresolved_point': None,
    'next_change': None,
    'change_reason': None,
}
SAVE = False
if SAVE:
    if not session.records:
        raise ValueError('保存する実行記録がありません。先に対話を実行してください。')
    if any(value is not None and not isinstance(value, str) for value in annotations.values()):
        raise ValueError('記入欄は文字列かNoneにしてください。')
    session.records[-1].update({key: value for key, value in annotations.items() if value is not None})
    paths = save_pair(session.records, ROOT / 'outputs')
    print([p.name for p in paths])
    assert load_records(paths[0]) == session.records
    assert load_records(paths[1]) == session.records
    # Colabのファイル欄からoutputs内の2ファイルをダウンロード。
else:
    print('保存は未実行です。前の記入内容も変更していません。')


## 次の質問・意図した反復
同じrun_idで送信セルを再実行すると拒否します。次の質問または確認済みの再試行をする場合だけ、次のセルを有効にして新しいrun_idを作り、質問・新規/継続・プレビューへ戻ります。別の学生は新しいランタイムを使います。

In [ ]:
START_NEXT_RUN = False
if START_NEXT_RUN:
    run_id = str(uuid4())
    print('新しいrunを準備しました。質問・新規/継続・プレビューを見直してください。')